# 01 — Pipeline overview

This notebook walks through the **Thinking on the Move (ToM)** pipeline using a
fully synthetic toy dataset — **no Cuebiq, no SafeGraph, no GPU required**.
It's the fastest way to grasp what the framework does end-to-end.

We will:
1. Inspect the toy data shape (workers, POIs, visits)
2. Build a single ToM-style prompt
3. Score the same visit with one classical baseline (gravity)
4. Show where the real pipeline differs (LoRA-Llama at scale)


## 1. Inspect the toy data

Schema mirrors the production pipeline.

In [ ]:
import json
import pandas as pd
from pathlib import Path

DATA = Path('../data/synthetic')
pois = pd.read_csv(DATA / 'toy_pois.csv')
visits = pd.read_csv(DATA / 'toy_visits.csv')

print(f'POIs: {len(pois)} across {pois.sub_category.nunique()} categories')
print(f'Visits: {len(visits)} from {visits.cuebiq_id.nunique()} workers'
      f' across {visits.cluster.nunique()} demographic clusters')
pois.head()

In [ ]:
visits.head()

## 2. A single ToM-style prompt

In [ ]:
train = [json.loads(l) for l in open(DATA / 'toy_train.jsonl')]
ex = train[0]
print('SYSTEM:'); print(ex['system'])
print()
print('USER:'); print(ex['user'][:1200], '...' if len(ex['user']) > 1200 else '')
print()
print('ASSISTANT (target):'); print(ex['assistant'][:600])

## 3. Classical baseline: gravity score

Even without an LLM, the candidate set + observed frequencies are enough
to fit a plain gravity model. We score the same visit's candidates here.

In [ ]:
import math, sys
sys.path.insert(0, '../src')
from tom.utils import haversine_m

def gravity_score(cand, alpha=1.0, beta=0.7):
    return (cand['cluster_freq'] ** alpha) * math.exp(-cand['dist_m'] / 800.0) ** beta

cands = ex['meta']['candidates']
ranked = sorted(cands, key=gravity_score, reverse=True)
true_id = ex['meta']['true_poi_id']
print('Truth:', true_id.split('|')[0])
print('Gravity top-5:')
for r in ranked[:5]:
    print(f"  {r['poi_name']:<32}  dist={r['dist_m']:>6.0f}m  freq={r['cluster_freq']}")

## 4. What the real pipeline adds

In the production setting:
- The candidate set comes from a real SafeGraph snapshot at the visit's week
- The persona text is a sample from a 15 000-persona pool clustered by ACS demographics
- The model is `Llama-3.1-8B + LoRA r=16` — fine-tuned for one epoch on 1 M visits
- Inference runs through vLLM in batches of 256 prompts on an H200

To run real inference on the toy set with the released DTBK adapter:

```bash
huggingface-cli download MazelTovy/thinking-on-the-move-dtbk --local-dir ./dtbk_adapter
python -m tom.core.infer_vllm \
    --records_in ../data/synthetic/toy_inference_records.jsonl \
    --model_path meta-llama/Llama-3.1-8B-Instruct \
    --lora_path ./dtbk_adapter \
    --out predictions.jsonl \
    --max_model_len 4096 --temperature 0.2
```

Continue with `02_dtbk_walkthrough.ipynb` for the DTBK story or `03_nyc_scaling.ipynb`
for the NYC headline numbers.